# 🏝️ Caribbean Capital EA — Genetic Evolution Engine
**Train your CCCi-based EA on T4 GPU (free!) to grow $250 → prop firm target**

- Evolves 7 per-symbol parameters: bull/bear levels, ATR stops, TP ratio, holding bars
- **ALSO evolves CCCi indicator weights** to find the best composite for each market
- Prop-firm-safe objective: maximise PF, hard cap on drawdown ≤ 10%
- Saves `evolved_profiles.json` for direct use in MT5

### Symbols: XAUUSD · GER40 · NAS100 · US30 · US500 · XAGUSD · BTCUSD


In [ ]:
# ─── Install dependencies ─────────────────────────────────────────────────────
!pip install -q numpy pandas matplotlib scipy tqdm

In [ ]:
# ─── Mount Google Drive (to access your M5 CSV files) ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, pathlib

# ───────────────────────────────────────────────────────────────────────────────
# OPTION A: point to where your CSVs live in Drive
# Change this path to match YOUR Google Drive folder
DRIVE_DATA_PATH = '/content/drive/MyDrive/caribbean_capital_data'

DATA_DIR = pathlib.Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

# Copy any CSV files from Drive into /content/data/
if os.path.isdir(DRIVE_DATA_PATH):
    copied = 0
    for f in os.listdir(DRIVE_DATA_PATH):
        if f.endswith('.csv') and '_M5' in f:
            shutil.copy2(os.path.join(DRIVE_DATA_PATH, f), DATA_DIR / f)
            print(f'  ✅ Copied {f}')
            copied += 1
    print(f'Copied {copied} M5 CSV files from Drive')
else:
    print(f'⚠️  Drive path not found: {DRIVE_DATA_PATH}')
    print('   → Either upload CSVs manually (next cell) or fix the path above')

In [ ]:
# ─── OR: Manual file upload (if Drive mount didn't work) ─────────────────────
# Run this cell if you want to upload CSV files from your computer instead

from google.colab import files
import pathlib

DATA_DIR = pathlib.Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

print('Select your M5 CSV files (XAUUSD_M5.csv, GER40_M5.csv, etc.)...')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = DATA_DIR / fname
    dest.write_bytes(data)
    print(f'  ✅ Saved {fname} ({len(data):,} bytes)')

print('Files in /content/data/:')
for f in sorted(DATA_DIR.glob('*.csv')):
    print(f'  {f.name}')

In [ ]:
# ─── Check which symbols are available ────────────────────────────────────────
import pathlib, os

DATA_DIR = pathlib.Path('/content/data')
SYMBOLS_ALL = ['XAUUSD', 'GER40', 'NAS100', 'US30', 'US500', 'XAGUSD', 'BTCUSD']

available = [s for s in SYMBOLS_ALL if (DATA_DIR / f'{s}_M5.csv').exists()]
missing   = [s for s in SYMBOLS_ALL if s not in available]

print(f'✅ Available ({len(available)}): {available}')
if missing:
    print(f'⚠️  Missing  ({len(missing)}): {missing}')
    print('   Missing symbols will be skipped during evolution')

SYMBOLS = available  # Only evolve symbols we have data for

In [ ]:
# ─── Caribbean Capital — Core Engine (validate + evolve) ──────────────────────
from __future__ import annotations

import copy, json, random, time, math
from dataclasses import dataclass, asdict, replace, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# ── paths ──────────────────────────────────────────────────────────────────────
DATA_DIR    = Path('/content/data')
OUTPUTS_DIR = Path('/content/outputs/caribbean_capital')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ── EAConfig dataclass ─────────────────────────────────────────────────────────
@dataclass
class EAConfig:
    risk_fraction: float = 0.002
    daily_trade_quota_max: int = 2
    daily_loss_limit: float = 0.02
    emergency_max_drawdown: float = 0.30
    max_lots: float = 0.30
    ccci_bull_level: float = 65.0
    ccci_bear_level: float = 35.0
    require_crossover: bool = True
    mfi40: int = 40
    mfi14: int = 14
    mfi20: int = 20
    mfi28: int = 28
    rsi21: int = 21
    rsi14: int = 14
    rsi8: int = 8
    rsi18: int = 18
    stoch_k28: int = 28
    stoch_k9: int = 9
    stoch_d: int = 3
    stoch_slow: int = 3
    bb60: int = 60
    bb30: int = 30
    bb20: int = 20
    bb_dev: float = 2.0
    wr14: int = 14
    wr28: int = 28
    cci10: int = 10
    cci60: int = 60
    vcb8: int = 8
    vcb14: int = 14
    lsp21: int = 21
    mtf_ent8: int = 8
    rfi14: int = 14
    atr_period: int = 14
    signal_period: int = 3
    adx_period: int = 14
    stop_atr: float = 1.5
    trail_atr: float = 0.8
    tp1_rr: float = 1.5
    max_holding_bars: int = 24
    asia_start_hour: int = 0
    asia_end_hour: int = 8
    london_start_hour: int = 7
    london_end_hour: int = 16
    ny_start_hour: int = 13
    ny_end_hour: int = 22
    account_type: int = 0
    adaptive_enable: bool = True
    adaptive_lookback: int = 20
    hot_win_rate: float = 0.60
    cold_win_rate: float = 0.40
    hot_risk_mult: float = 1.25
    cold_risk_mult: float = 0.55
    danger_risk_mult: float = 0.25
    adaptive_atr_widen: float = 1.30
    adaptive_atr_tighten: float = 0.85
    prop_daily_dd: float = 0.05
    prop_max_total_dd: float = 0.10
    prop_profit_target: float = 0.08
    prop_consistency_cap: float = 0.30
    prop_weekend_flat: bool = True
    prop_no_overnight: bool = False
    prop_max_lot_mult: float = 1.0
    prop_entry_buffer: int = 2
    # CCCi indicator weights (evolvable)
    w_mfi: float = 0.28
    w_rsi: float = 0.20
    w_stoch: float = 0.15
    w_bb: float = 0.12
    w_wr: float = 0.10
    w_cci: float = 0.08
    w_vcb: float = 0.04
    w_lsp: float = 0.015
    w_mtf: float = 0.010
    w_rfi: float = 0.005

# ── indicator helpers ──────────────────────────────────────────────────────────
def clamp1(values):
    return pd.Series(np.clip(np.asarray(values, dtype=float), -1.0, 1.0))

def n100(values):
    return (values - 50.0) / 50.0

def norm_wpr(values):
    return -(values + 50.0) / 50.0

def norm_cci(values):
    return pd.Series(np.clip(values / 200.0, -1.0, 1.0), index=values.index)

def rma(series, period):
    return series.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()

def ema(series, period):
    return series.ewm(span=period, adjust=False, min_periods=1).mean()

def atr_ind(df, period):
    prev_close = df["close"].shift(1)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - prev_close).abs(),
        (df["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return rma(tr, period)

def rsi_ind(series, period):
    delta = series.diff()
    up   = delta.clip(lower=0.0)
    down = -delta.clip(upper=0.0)
    rs   = rma(up, period) / rma(down, period).replace(0.0, np.nan)
    return (100.0 - (100.0 / (1.0 + rs))).fillna(50.0)

def mfi_ind(df, period):
    typical    = (df["high"] + df["low"] + df["close"]) / 3.0
    money_flow = typical * df["volume"].fillna(0.0)
    delta      = typical.diff()
    pos_sum    = money_flow.where(delta > 0.0, 0.0).rolling(period, min_periods=period).sum()
    neg_sum    = money_flow.where(delta < 0.0, 0.0).rolling(period, min_periods=period).sum()
    ratio      = pos_sum / neg_sum.replace(0.0, np.nan)
    return (100.0 - (100.0 / (1.0 + ratio))).fillna(50.0)

def stochastic_ind(df, k_period, d_period, slowing):
    lowest  = df["low"].rolling(k_period, min_periods=k_period).min()
    highest = df["high"].rolling(k_period, min_periods=k_period).max()
    span    = (highest - lowest).replace(0.0, np.nan)
    main    = (100.0 * (df["close"] - lowest) / span).rolling(slowing, min_periods=slowing).mean()
    signal  = main.rolling(d_period, min_periods=d_period).mean()
    return main.fillna(50.0), signal.fillna(50.0)

def bb_zscore(close, period, dev):
    mid  = close.rolling(period, min_periods=period).mean()
    std  = close.rolling(period, min_periods=period).std(ddof=0)
    z    = (close - mid) / (std * dev).replace(0.0, np.nan)
    return z.fillna(0.0)

def wpr_ind(df, period):
    highest = df["high"].rolling(period, min_periods=period).max()
    lowest  = df["low"].rolling(period, min_periods=period).min()
    span    = (highest - lowest).replace(0.0, np.nan)
    return (-100.0 * (highest - df["close"]) / span).fillna(-50.0)

def cci_ind(df, period):
    typical  = (df["high"] + df["low"] + df["close"]) / 3.0
    sma      = typical.rolling(period, min_periods=period).mean()
    mean_dev = (typical - sma).abs().rolling(period, min_periods=period).mean()
    return ((typical - sma) / (0.015 * mean_dev.replace(0.0, np.nan))).fillna(0.0)

def lsp_ind(df, lookback):
    recent_high = df["high"].shift(1).rolling(lookback, min_periods=lookback).max()
    recent_low  = df["low"].shift(1).rolling(lookback, min_periods=lookback).min()
    span  = recent_high - recent_low
    score = 2.0 * (df["close"] - recent_low) / span.replace(0.0, np.nan) - 1.0
    score = score.where(df["close"] <= recent_high, 1.0)
    score = score.where(df["close"] >= recent_low, -1.0)
    return pd.Series(np.clip(score.fillna(0.0), -1.0, 1.0), index=df.index)

def mtf_entropy_ind(close, base):
    s1 = np.where(close > close.shift(base),     1.0, -1.0)
    s2 = np.where(close > close.shift(base * 2), 1.0, -1.0)
    s3 = np.where(close > close.shift(base * 4), 1.0, -1.0)
    return pd.Series(np.clip((s1 + s2 + s3) / 3.0, -1.0, 1.0), index=close.index)

def regime_fracture_ind(close, lookback):
    total_move  = close.diff(lookback).abs()
    noise_path  = close.diff().abs().rolling(lookback, min_periods=lookback).sum()
    direction   = np.where(close > close.shift(lookback), 1.0, -1.0)
    out = direction * (total_move / noise_path.replace(0.0, np.nan))
    return pd.Series(np.clip(pd.Series(out, index=close.index).fillna(0.0), -1.0, 1.0), index=close.index)

def adx_ind(df, period=14):
    h, l, c = df["high"], df["low"], df["close"]
    plus_dm  = np.where(((h - h.shift(1)) > (l.shift(1) - l)) & ((h - h.shift(1)) > 0), h - h.shift(1), 0.0)
    minus_dm = np.where(((l.shift(1) - l) > (h - h.shift(1))) & ((l.shift(1) - l) > 0), l.shift(1) - l, 0.0)
    tr       = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
    atr_s    = rma(tr, period)
    plus_di  = 100.0 * rma(pd.Series(plus_dm, index=df.index), period) / atr_s.replace(0.0, np.nan)
    minus_di = 100.0 * rma(pd.Series(minus_dm, index=df.index), period) / atr_s.replace(0.0, np.nan)
    dx       = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)
    return rma(dx.fillna(0.0), period).fillna(0.0)

# ── CCCi computation ───────────────────────────────────────────────────────────
def compute_ccci(df: pd.DataFrame, cfg: EAConfig) -> pd.DataFrame:
    out = df.copy()
    out["atr"]   = atr_ind(out, cfg.atr_period)
    out["mfi40"] = mfi_ind(out, cfg.mfi40);   out["mfi14"] = mfi_ind(out, cfg.mfi14)
    out["mfi20"] = mfi_ind(out, cfg.mfi20);   out["mfi28"] = mfi_ind(out, cfg.mfi28)
    out["rsi21"] = rsi_ind(out["close"], cfg.rsi21); out["rsi14"] = rsi_ind(out["close"], cfg.rsi14)
    out["rsi8"]  = rsi_ind(out["close"], cfg.rsi8);  out["rsi18"] = rsi_ind(out["close"], cfg.rsi18)
    out["sk28"], out["sd28"] = stochastic_ind(out, cfg.stoch_k28, cfg.stoch_d, cfg.stoch_slow)
    out["sk9"],  out["sd9"]  = stochastic_ind(out, cfg.stoch_k9,  cfg.stoch_d, cfg.stoch_slow)
    out["bbz60"] = bb_zscore(out["close"], cfg.bb60, cfg.bb_dev)
    out["bbz30"] = bb_zscore(out["close"], cfg.bb30, cfg.bb_dev)
    out["bbz20"] = bb_zscore(out["close"], cfg.bb20, cfg.bb_dev)
    out["wr14"]  = wpr_ind(out, cfg.wr14);   out["wr28"]  = wpr_ind(out, cfg.wr28)
    out["cci10"] = cci_ind(out, cfg.cci10);  out["cci60"] = cci_ind(out, cfg.cci60)
    out["adx"]   = adx_ind(out, cfg.adx_period)

    atr_min8  = out["atr"].shift(1).rolling(cfg.vcb8,  min_periods=cfg.vcb8).min()
    atr_min14 = out["atr"].shift(1).rolling(cfg.vcb14, min_periods=cfg.vcb14).min()
    exp8   = ((out["atr"] / atr_min8.replace(0.0,  np.nan)) - 1.0).clip(upper=1.5) / 1.5
    exp14  = ((out["atr"] / atr_min14.replace(0.0, np.nan)) - 1.0).clip(upper=1.5) / 1.5
    mom8   = np.where(out["close"] > out["close"].shift(cfg.vcb8),  1.0, -1.0)
    mom14  = np.where(out["close"] > out["close"].shift(cfg.vcb14), 1.0, -1.0)
    vcb8_s  = np.clip(exp8.fillna(0.0)  * mom8,  -1.0, 1.0)
    vcb14_s = np.clip(exp14.fillna(0.0) * mom14, -1.0, 1.0)

    tf = np.clip((out["adx"] - 25.0) / 25.0, 0.0, 1.0)
    rf = 1.0 - tf

    mfi_score   = np.clip(n100(out["mfi40"])*0.40 + n100(out["mfi14"])*0.30 + n100(out["mfi20"])*0.17 + n100(out["mfi28"])*0.13, -1.0, 1.0)
    rsi_score   = np.clip(n100(out["rsi21"])*0.40 + n100(out["rsi14"])*0.30 + n100(out["rsi8"])*0.18  + n100(out["rsi18"])*0.12,  -1.0, 1.0)
    stoch_score = np.clip(n100(out["sk28"])*0.35  + n100(out["sk9"])*0.35   + n100(out["sd9"])*0.17   + n100(out["sd28"])*0.13,   -1.0, 1.0)
    raw_bbz     = np.clip(out["bbz60"]*0.50 + out["bbz30"]*0.30 + out["bbz20"]*0.20, -1.0, 1.0)
    bb_score    = np.clip(raw_bbz * (2.0 * tf - 1.0), -1.0, 1.0)
    raw_wr      = np.clip(norm_wpr(out["wr14"])*0.60 + norm_wpr(out["wr28"])*0.40, -1.0, 1.0)
    wr_score    = np.clip(raw_wr * (rf - tf), -1.0, 1.0)
    cci_score   = np.clip(norm_cci(out["cci10"])*0.55 + norm_cci(out["cci60"])*0.45, -1.0, 1.0)
    vcb_score   = np.clip(vcb8_s * 0.60 + vcb14_s * 0.40, -1.0, 1.0)
    lsp_score   = lsp_ind(out, cfg.lsp21)
    mtf_score   = mtf_entropy_ind(out["close"], cfg.mtf_ent8)
    rfi_score   = regime_fracture_ind(out["close"], cfg.rfi14)

    # Use evolved weights (normalise so they sum to 1.0)
    ww = cfg.w_mfi + cfg.w_rsi + cfg.w_stoch + cfg.w_bb + cfg.w_wr + cfg.w_cci + cfg.w_vcb + cfg.w_lsp + cfg.w_mtf + cfg.w_rfi
    if ww <= 0:
        ww = 1.0
    composite = (
        cfg.w_mfi   / ww * mfi_score
        + cfg.w_rsi   / ww * rsi_score
        + cfg.w_stoch / ww * stoch_score
        + cfg.w_bb    / ww * bb_score
        + cfg.w_wr    / ww * wr_score
        + cfg.w_cci   / ww * cci_score
        + cfg.w_vcb   / ww * vcb_score
        + cfg.w_lsp   / ww * lsp_score
        + cfg.w_mtf   / ww * mtf_score
        + cfg.w_rfi   / ww * rfi_score
    )
    out["ccci"]  = np.clip((composite + 1.0) * 50.0, 0.0, 100.0)
    out["signal"] = ema(out["ccci"], cfg.signal_period)
    out["warmup_ready"] = np.arange(len(out)) >= (max(cfg.mfi40, cfg.bb60) + max(cfg.lsp21, cfg.rfi14) + cfg.adx_period + 20)
    return out

# ── session filter ─────────────────────────────────────────────────────────────
def in_enabled_session(ts, cfg: EAConfig) -> bool:
    h   = int(ts.hour)
    m   = int(ts.minute)
    buf = cfg.prop_entry_buffer if cfg.account_type > 0 else 0
    if buf > 0:
        if h == cfg.asia_start_hour   and m < buf: return False
        if h == cfg.london_start_hour and m < buf: return False
        if h == cfg.ny_start_hour     and m < buf: return False
    return (cfg.asia_start_hour   <= h < cfg.asia_end_hour
         or cfg.london_start_hour <= h < cfg.london_end_hour
         or cfg.ny_start_hour     <= h < cfg.ny_end_hour)

# ── backtest engine ────────────────────────────────────────────────────────────
def backtest_symbol(df: pd.DataFrame, symbol: str, cfg: EAConfig) -> dict:
    equity = 1.0
    equity_curve: list = []
    trades: list = []
    position = None
    g_equity_day_start = 1.0
    g_phase_start_balance = 1.0
    g_all_time_high = 1.0
    g_trades_today = 0
    g_day_of_year = None
    g_regime = 0
    g_risk_multiplier = 1.0
    g_atr_multiplier = 1.0
    trade_wins = [-1] * max(5, cfg.adaptive_lookback)
    trade_ring_head = 0
    trade_ring_size = 0

    def record_trade_result(pnl_value):
        nonlocal trade_ring_head, trade_ring_size
        trade_wins[trade_ring_head] = 1 if pnl_value > 0.0 else 0
        trade_ring_head = (trade_ring_head + 1) % len(trade_wins)
        trade_ring_size = min(trade_ring_size + 1, len(trade_wins))

    def update_regime():
        nonlocal g_regime, g_risk_multiplier, g_atr_multiplier
        if not cfg.adaptive_enable or trade_ring_size < 5:
            return
        wins = sum(1 for v in trade_wins if v == 1)
        total = sum(1 for v in trade_wins if v >= 0)
        gross_w = sum(1.0 for v in trade_wins if v == 1) + 0.0001
        gross_l = sum(1.0 for v in trade_wins if v == 0) + 0.0001
        win_rate = wins / total if total else 0.5
        pf_proxy = gross_w / gross_l
        cur_dd = (g_all_time_high - equity) / g_all_time_high if g_all_time_high > 0.0 else 0.0
        if cur_dd >= cfg.emergency_max_drawdown * 0.70:
            g_regime = 3; g_risk_multiplier = cfg.danger_risk_mult;  g_atr_multiplier = cfg.adaptive_atr_widen
        elif win_rate >= cfg.hot_win_rate and pf_proxy >= 1.5:
            g_regime = 1; g_risk_multiplier = cfg.hot_risk_mult;   g_atr_multiplier = cfg.adaptive_atr_tighten
        elif win_rate <= cfg.cold_win_rate or pf_proxy < 1.0:
            g_regime = 2; g_risk_multiplier = cfg.cold_risk_mult;  g_atr_multiplier = cfg.adaptive_atr_widen
        else:
            g_regime = 0; g_risk_multiplier = 1.0; g_atr_multiplier = 1.0

    for idx in range(2, len(df)):
        row = df.iloc[idx]
        ts  = df.index[idx]
        day_of_year = int(ts.day_of_year)

        if g_day_of_year != day_of_year:
            g_day_of_year = day_of_year
            g_trades_today = 0
            g_equity_day_start = equity
            g_all_time_high = max(g_all_time_high, equity)

        if position is not None:
            position["bars_held"] = int(position["bars_held"]) + 1
            trail = float(df.iloc[idx - 1]["atr"]) * cfg.trail_atr * g_atr_multiplier
            if position["side"] == 1:
                position["stop"] = max(float(position["stop"]), float(row["open"]) - trail)
            else:
                position["stop"] = min(float(position["stop"]), float(row["open"]) + trail)

            exit_reason = exit_price = None
            if int(position["bars_held"]) >= cfg.max_holding_bars:
                exit_reason = "time";  exit_price = float(row["open"])
            elif position["side"] == 1:
                if float(row["low"])  <= float(position["stop"]): exit_reason = "stop"; exit_price = float(position["stop"])
                elif float(row["high"]) >= float(position["tp"]): exit_reason = "tp";   exit_price = float(position["tp"])
            else:
                if float(row["high"]) >= float(position["stop"]): exit_reason = "stop"; exit_price = float(position["stop"])
                elif float(row["low"]) <= float(position["tp"]):  exit_reason = "tp";   exit_price = float(position["tp"])

            if exit_reason is not None:
                risk_distance = abs(float(position["entry"]) - float(position["initial_stop"]))
                pnl_r = 0.0 if risk_distance <= 0.0 else int(position["side"]) * (float(exit_price) - float(position["entry"])) / risk_distance
                equity *= 1.0 + float(position["risk_fraction"]) * pnl_r
                g_all_time_high = max(g_all_time_high, equity)
                trades.append({"symbol": symbol, "entry_time": str(position["entry_time"]), "exit_time": str(ts),
                                "side": "long" if int(position["side"]) == 1 else "short",
                                "entry_price": float(position["entry"]), "exit_price": float(exit_price),
                                "pnl_r": float(pnl_r), "exit_reason": exit_reason})
                record_trade_result(pnl_r)
                update_regime()
                position = None

        daily_loss_breached = g_equity_day_start > 0.0 and ((g_equity_day_start - equity) / g_equity_day_start) >= cfg.daily_loss_limit
        emergency_dd_breached = g_phase_start_balance > 0.0 and ((g_phase_start_balance - equity) / g_phase_start_balance) >= cfg.emergency_max_drawdown
        if position is None and not daily_loss_breached and not emergency_dd_breached:
            if row["warmup_ready"] and g_trades_today < cfg.daily_trade_quota_max and in_enabled_session(ts, cfg):
                ccci_now  = float(df.iloc[idx - 1]["ccci"])
                ccci_prev = float(df.iloc[idx - 2]["ccci"])
                signal = 0
                if cfg.require_crossover:
                    if ccci_prev <= cfg.ccci_bull_level and ccci_now > cfg.ccci_bull_level: signal = 1
                    elif ccci_prev >= cfg.ccci_bear_level and ccci_now < cfg.ccci_bear_level: signal = -1
                else:
                    if ccci_now > cfg.ccci_bull_level: signal = 1
                    elif ccci_now < cfg.ccci_bear_level: signal = -1

                atr_prev = float(df.iloc[idx - 1]["atr"])
                if signal != 0 and np.isfinite(atr_prev) and atr_prev > 0.0:
                    eff_stop = cfg.stop_atr * g_atr_multiplier
                    entry = float(row["open"])
                    stop  = entry - atr_prev * eff_stop if signal > 0 else entry + atr_prev * eff_stop
                    tp    = entry + atr_prev * eff_stop * cfg.tp1_rr if signal > 0 else entry - atr_prev * eff_stop * cfg.tp1_rr
                    position = {"entry_time": ts, "side": signal, "entry": entry,
                                "stop": stop, "initial_stop": stop, "tp": tp,
                                "bars_held": 0, "risk_fraction": cfg.risk_fraction * g_risk_multiplier}
                    g_trades_today += 1

        equity_curve.append(equity)

    equity_series = pd.Series(equity_curve, index=df.index[2:])
    if equity_series.empty or not trades:
        return {"metrics": {"symbol": symbol, "net_return": 0.0, "profit_factor": 0.0,
                            "max_drawdown": 0.0, "win_rate": 0.0, "trades": 0},
                "trades": [], "equity_curve": equity_curve}

    t_df = pd.DataFrame(trades)
    gross_profit = t_df.loc[t_df["pnl_r"] > 0.0, "pnl_r"].sum()
    gross_loss   = t_df.loc[t_df["pnl_r"] < 0.0, "pnl_r"].sum()
    drawdown     = (equity_series / equity_series.cummax()) - 1.0
    return {
        "metrics": {
            "symbol": symbol,
            "net_return": float(equity_series.iloc[-1] - 1.0),
            "profit_factor": float(gross_profit / abs(gross_loss)) if gross_loss != 0.0 else 0.0,
            "max_drawdown": float(abs(drawdown.min())),
            "win_rate": float((t_df["pnl_r"] > 0.0).mean()),
            "trades": len(t_df),
        },
        "trades": trades,
        "equity_curve": equity_curve,
    }

# ── data loader ────────────────────────────────────────────────────────────────
def load_symbol_data(symbol: str, data_dir=None) -> pd.DataFrame:
    d = Path(data_dir) if data_dir else DATA_DIR
    path = d / f"{symbol}_M5.csv"
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    df = df.sort_values("datetime").set_index("datetime")
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.dropna(subset=["open", "high", "low", "close"])

print("✅ Caribbean Capital Engine loaded — ready to evolve")

In [ ]:
# ─── Genetic Evolution Engine (with CCCi weight evolution) ───────────────────

# ── parameter + weight bounds ──────────────────────────────────────────────────
EA_BOUNDS: dict[str, tuple[float, float]] = {
    "ccci_bull_level":  (55.0, 80.0),
    "ccci_bear_level":  (20.0, 45.0),
    "stop_atr":         (0.8,  2.5),
    "trail_atr":        (0.4,  1.8),
    "tp1_rr":           (0.8,  3.5),
    "max_holding_bars": (6.0,  72.0),
}

# CCCi indicator weight bounds — allow evolution to redistribute importance
WEIGHT_BOUNDS: dict[str, tuple[float, float]] = {
    "w_mfi":   (0.05, 0.50),
    "w_rsi":   (0.05, 0.45),
    "w_stoch": (0.02, 0.40),
    "w_bb":    (0.02, 0.35),
    "w_wr":    (0.01, 0.30),
    "w_cci":   (0.01, 0.30),
    "w_vcb":   (0.01, 0.25),
    "w_lsp":   (0.001, 0.15),
    "w_mtf":   (0.001, 0.20),
    "w_rfi":   (0.001, 0.10),
}

ALL_BOUNDS = {**EA_BOUNDS, **WEIGHT_BOUNDS}

def _random_gene(rng, evolve_weights=True) -> dict:
    gene = {}
    for key, (lo, hi) in EA_BOUNDS.items():
        gene[key] = rng.uniform(lo, hi)
    gene["require_crossover"] = rng.random() < 0.6
    if evolve_weights:
        for key, (lo, hi) in WEIGHT_BOUNDS.items():
            gene[key] = rng.uniform(lo, hi)
    else:
        for key in WEIGHT_BOUNDS:
            gene[key] = getattr(EAConfig(), key)
    return gene

def _clamp(gene: dict) -> dict:
    out = dict(gene)
    for key, (lo, hi) in ALL_BOUNDS.items():
        out[key] = max(lo, min(hi, out.get(key, lo)))
    out["ccci_bull_level"] = max(51.0, out["ccci_bull_level"])
    out["ccci_bear_level"] = min(49.0, out["ccci_bear_level"])
    spread = out["ccci_bull_level"] + (100.0 - out["ccci_bear_level"])
    if spread < 115.0:
        deficit = 115.0 - spread
        out["ccci_bull_level"] = min(80.0, out["ccci_bull_level"] + deficit / 2.0)
        out["ccci_bear_level"] = max(20.0, out["ccci_bear_level"] - deficit / 2.0)
    return out

def _gene_to_cfg(base: EAConfig, gene: dict) -> EAConfig:
    return replace(base,
        ccci_bull_level=gene["ccci_bull_level"],
        ccci_bear_level=gene["ccci_bear_level"],
        stop_atr=gene["stop_atr"], trail_atr=gene["trail_atr"],
        tp1_rr=gene["tp1_rr"], max_holding_bars=int(round(gene["max_holding_bars"])),
        require_crossover=bool(gene["require_crossover"]),
        w_mfi=gene.get("w_mfi", base.w_mfi),   w_rsi=gene.get("w_rsi", base.w_rsi),
        w_stoch=gene.get("w_stoch", base.w_stoch), w_bb=gene.get("w_bb", base.w_bb),
        w_wr=gene.get("w_wr", base.w_wr),       w_cci=gene.get("w_cci", base.w_cci),
        w_vcb=gene.get("w_vcb", base.w_vcb),    w_lsp=gene.get("w_lsp", base.w_lsp),
        w_mtf=gene.get("w_mtf", base.w_mtf),    w_rfi=gene.get("w_rfi", base.w_rfi),
    )

def _fitness(metrics: dict, pf_floor: float, dd_cap: float, min_trades: int) -> float:
    pf     = float(metrics["profit_factor"])
    dd     = float(metrics["max_drawdown"])
    net    = float(metrics["net_return"])
    trades = int(metrics["trades"])
    if dd > dd_cap:
        return -10.0 - (dd - dd_cap) * 50.0
    value = pf * 6.0 + net * 3.0 - dd * 20.0
    if pf < pf_floor:
        value -= (pf_floor - pf) * 8.0
    if trades < min_trades:
        value -= 3.0 * (1.0 - trades / max(min_trades, 1))
    return value

def _crossover(pa, pb, rng, alpha=0.5) -> dict:
    child = {}
    for key in ALL_BOUNDS:
        a, b = pa.get(key, 0.0), pb.get(key, 0.0)
        lo = min(a, b) - alpha * abs(b - a)
        hi = max(a, b) + alpha * abs(b - a)
        child[key] = rng.uniform(lo, hi)
    child["require_crossover"] = pa["require_crossover"] if rng.random() < 0.5 else pb["require_crossover"]
    return _clamp(child)

def _mutate(gene, rng, mutation_rate, sigma_frac) -> dict:
    out = dict(gene)
    for key, (lo, hi) in ALL_BOUNDS.items():
        if rng.random() < mutation_rate:
            sigma = (hi - lo) * sigma_frac
            out[key] = out.get(key, lo) + rng.gauss(0.0, sigma)
    if rng.random() < mutation_rate:
        out["require_crossover"] = not out["require_crossover"]
    return _clamp(out)

def _tournament(pop, scores, k, rng) -> dict:
    indices = rng.sample(range(len(pop)), k)
    return copy.deepcopy(pop[max(indices, key=lambda i: scores[i])])

# ── main evolution loop per symbol ────────────────────────────────────────────
def evolve_symbol(symbol, prepared, base, *, pop_size, generations, pf_floor, dd_cap,
                   min_trades, elitism, mutation_rate, sigma_frac, tournament_k,
                   seed_genes, rng, verbose, evolve_weights=True):
    pop = [_clamp(g) for g in (seed_genes or [])]
    while len(pop) < pop_size:
        pop.append(_random_gene(rng, evolve_weights))
    pop = pop[:pop_size]

    def evaluate(g):
        cfg = _gene_to_cfg(base, g)
        result = backtest_symbol(prepared, symbol, cfg)
        m = result["metrics"]
        return _fitness(m, pf_floor, dd_cap, min_trades), m, result.get("equity_curve", [])

    scored = [evaluate(g) for g in pop]
    scores = [s for s, _, _ in scored]
    best_score = max(scores)
    best_idx = scores.index(best_score)
    best_gene = copy.deepcopy(pop[best_idx])
    best_metrics = scored[best_idx][1]
    best_equity_curve = scored[best_idx][2]

    mr = mutation_rate
    for gen in range(generations):
        order = sorted(range(len(pop)), key=lambda i: scores[i], reverse=True)
        new_pop = [copy.deepcopy(pop[order[i]]) for i in range(min(elitism, len(order)))]
        while len(new_pop) < pop_size:
            child = _crossover(_tournament(pop, scores, tournament_k, rng),
                                _tournament(pop, scores, tournament_k, rng), rng)
            new_pop.append(_mutate(child, rng, mr, sigma_frac))
        pop = new_pop
        scored = [evaluate(g) for g in pop]
        scores = [s for s, _, _ in scored]
        gen_best_idx = scores.index(max(scores))
        if scores[gen_best_idx] > best_score:
            best_score = scores[gen_best_idx]
            best_gene = copy.deepcopy(pop[gen_best_idx])
            best_metrics = scored[gen_best_idx][1]
            best_equity_curve = scored[gen_best_idx][2]
        diversity = max(scores) - min(scores)
        if diversity < 0.5 and mr < 0.5:
            mr = min(0.5, mr * 1.2)
        if verbose and (gen + 1) % max(1, generations // 10) == 0:
            m = best_metrics
            print(f"  [{symbol}] gen {gen+1:3d}/{generations} | score={best_score:.3f} | "
                  f"PF={float(m['profit_factor']):.3f} | DD={float(m['max_drawdown']):.4f} | "
                  f"trades={m['trades']}")

    return best_gene, best_metrics, best_equity_curve

print("✅ Genetic Evolution Engine loaded")

In [ ]:
# ─── Evolution Config — Tune Here ─────────────────────────────────────────────
# Prop-firm-safe defaults — adjust to taste

POP_SIZE      = 40      # Population size per symbol (↑ = better results, slower)
GENERATIONS   = 35      # Generations per symbol (↑ = better results, slower)
PF_FLOOR      = 1.70    # Minimum acceptable Profit Factor
DD_CAP        = 0.10    # Maximum drawdown allowed (10%)
MIN_TRADES    = 30      # Minimum trades required for a valid solution
ELITISM       = 3       # Top N elites survive each generation unchanged
MUTATION_RATE = 0.20    # Starting mutation probability per gene
SIGMA_FRAC    = 0.10    # Gaussian mutation width as fraction of parameter range
TOURNAMENT_K  = 3       # Tournament selection pool size
SEED          = 42      # Random seed for reproducibility

EVOLVE_WEIGHTS = True   # Set True to also evolve CCCi indicator weights
                        # This discovers which indicators matter most per symbol

# Symbols to evolve (auto-set from available data above)
# Override here if you want to run fewer symbols:
# SYMBOLS = ['XAUUSD', 'GER40']

print(f"Config: pop={POP_SIZE} | gens={GENERATIONS} | pf_floor={PF_FLOOR} | dd_cap={DD_CAP}")
print(f"Evolve CCCi weights: {EVOLVE_WEIGHTS}")
print(f"Symbols: {SYMBOLS}")
estimated_backtests = len(SYMBOLS) * GENERATIONS * POP_SIZE
print(f"Est. backtests: ~{estimated_backtests:,}")

In [ ]:
# ─── 🚀 Run Evolution — This is the main training cell ────────────────────────
import time

rng  = random.Random(SEED)
base = EAConfig()

profiles    = {}
all_results = []
all_curves  = {}

total_start = time.time()

for symbol in SYMBOLS:
    print(f"\n{'─'*60}")
    print(f"🧬 Evolving {symbol}...")
    t0 = time.time()

    try:
        raw_data = load_symbol_data(symbol)
        prepared = compute_ccci(raw_data, base)
    except Exception as e:
        print(f"  ⚠️  Could not load {symbol}: {e}")
        continue

    print(f"  Data: {len(prepared):,} bars | {prepared.index[0].date()} → {prepared.index[-1].date()}")

    best_gene, best_metrics, equity_curve = evolve_symbol(
        symbol=symbol,
        prepared=prepared,
        base=base,
        pop_size=POP_SIZE,
        generations=GENERATIONS,
        pf_floor=PF_FLOOR,
        dd_cap=DD_CAP,
        min_trades=MIN_TRADES,
        elitism=ELITISM,
        mutation_rate=MUTATION_RATE,
        sigma_frac=SIGMA_FRAC,
        tournament_k=TOURNAMENT_K,
        seed_genes=None,
        rng=rng,
        verbose=True,
        evolve_weights=EVOLVE_WEIGHTS,
    )

    elapsed = time.time() - t0
    pf = float(best_metrics['profit_factor'])
    dd = float(best_metrics['max_drawdown'])
    net = float(best_metrics['net_return'])
    trades = int(best_metrics['trades'])
    status = 'PASS' if pf >= PF_FLOOR and dd <= DD_CAP else 'BELOW_TARGET'

    profile = {
        'ccci_bull_level':  round(best_gene['ccci_bull_level'], 2),
        'ccci_bear_level':  round(best_gene['ccci_bear_level'], 2),
        'require_crossover': bool(best_gene['require_crossover']),
        'stop_atr':         round(best_gene['stop_atr'], 3),
        'trail_atr':        round(best_gene['trail_atr'], 3),
        'tp1_rr':           round(best_gene['tp1_rr'], 3),
        'max_holding_bars': int(round(best_gene['max_holding_bars'])),
        'status': status,
    }
    if EVOLVE_WEIGHTS:
        profile['indicator_weights'] = {
            k: round(best_gene.get(k, getattr(base, k)), 4)
            for k in ['w_mfi','w_rsi','w_stoch','w_bb','w_wr','w_cci','w_vcb','w_lsp','w_mtf','w_rfi']
        }

    profiles[symbol] = profile
    all_curves[symbol] = equity_curve

    icon = '✅' if status == 'PASS' else '⚠️ '
    print(f"  {icon} {symbol}: PF={pf:.3f}  DD={dd:.4f}  net={net:.2%}  trades={trades}  [{status}]  ({elapsed:.0f}s)")
    print(f"     bull={profile['ccci_bull_level']}  bear={profile['ccci_bear_level']}  "
          f"stop={profile['stop_atr']}  trail={profile['trail_atr']}  "
          f"tp={profile['tp1_rr']}  hold={profile['max_holding_bars']}  "
          f"cross={profile['require_crossover']}")

total_elapsed = time.time() - total_start
passing = [s for s, p in profiles.items() if p['status'] == 'PASS']
below   = [s for s, p in profiles.items() if p['status'] != 'PASS']

print(f"\n{'═'*60}")
print(f"✅ PASS    ({len(passing)}): {passing}")
print(f"⚠️  BELOW   ({len(below)}): {below}")
print(f"Total time: {total_elapsed/60:.1f} min")

In [ ]:
# ─── Save evolved profiles ────────────────────────────────────────────────────
import json, pathlib

OUTPUTS_DIR = pathlib.Path('/content/outputs/caribbean_capital')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

payload = {
    'engine': 'CaribbeanCapital Genetic Evolution — Colab T4',
    'pf_floor': PF_FLOOR,
    'dd_cap': DD_CAP,
    'pop_size': POP_SIZE,
    'generations': GENERATIONS,
    'evolve_weights': EVOLVE_WEIGHTS,
    'passing_symbols': passing,
    'below_target_symbols': below,
    'profiles': profiles,
}

out_path = OUTPUTS_DIR / 'evolved_profiles.json'
out_path.write_text(json.dumps(payload, indent=2))
print(f'✅ Saved: {out_path}')

# Also save to Drive if mounted
try:
    import shutil
    drive_out = pathlib.Path('/content/drive/MyDrive/caribbean_capital_evolved_profiles.json')
    shutil.copy2(out_path, drive_out)
    print(f'✅ Saved to Drive: {drive_out}')
except Exception as e:
    print(f'   (Drive save skipped: {e})')

In [ ]:
# ─── Download evolved_profiles.json ──────────────────────────────────────────
from google.colab import files
files.download('/content/outputs/caribbean_capital/evolved_profiles.json')
print('📥 Download triggered — check your browser Downloads folder')

In [ ]:
# ─── 📊 Equity Curves — visualise each symbol's performance ─────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

n = len(all_curves)
if n == 0:
    print('No results to plot')
else:
    cols = min(3, n)
    rows = math.ceil(n / cols)
    fig = plt.figure(figsize=(7 * cols, 4 * rows))
    fig.suptitle('Caribbean Capital EA — Evolved Equity Curves', fontsize=14, fontweight='bold')

    for i, (symbol, curve) in enumerate(all_curves.items()):
        ax = fig.add_subplot(rows, cols, i + 1)
        eq = np.array(curve)
        peak = np.maximum.accumulate(eq)
        dd   = (eq - peak) / peak

        color = '#00c853' if profiles.get(symbol, {}).get('status') == 'PASS' else '#ff6d00'
        ax.fill_between(range(len(eq)), 1.0, eq, alpha=0.15, color=color)
        ax.plot(eq, color=color, linewidth=1.5, label='Equity')
        ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

        p = profiles.get(symbol, {})
        pf  = profiles[symbol].get('profit_factor', 0) if 'profit_factor' in profiles.get(symbol, {}) else 0
        status_icon = '✅' if p.get('status') == 'PASS' else '⚠️'
        ax.set_title(f'{status_icon} {symbol}', fontsize=10)
        ax.set_xlabel('Bars')
        ax.set_ylabel('Equity (normalised)')
        ax.grid(alpha=0.3)

        # Annotate final equity
        final_eq = eq[-1] if len(eq) > 0 else 1.0
        ax.annotate(f'+{(final_eq-1):.1%}', xy=(len(eq)-1, final_eq),
                    xytext=(-30, 10), textcoords='offset points',
                    fontsize=8, color=color, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/content/outputs/caribbean_capital/equity_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved equity curves to /content/outputs/caribbean_capital/equity_curves.png')

In [ ]:
# ─── 🔥 CCCi Indicator Weight Heatmap (if weights were evolved) ──────────────
import matplotlib.pyplot as plt
import numpy as np

if not EVOLVE_WEIGHTS:
    print('Weight evolution was disabled — set EVOLVE_WEIGHTS=True to see this chart')
else:
    weight_keys = ['w_mfi','w_rsi','w_stoch','w_bb','w_wr','w_cci','w_vcb','w_lsp','w_mtf','w_rfi']
    labels = ['MFI','RSI','Stoch','BB','WR','CCI','VCB','LSP','MTF-Entropy','Regime-Fracture']
    symbol_list = list(profiles.keys())

    matrix = []
    for sym in symbol_list:
        row_weights = profiles[sym].get('indicator_weights', {})
        raw = [row_weights.get(k, getattr(EAConfig(), k)) for k in weight_keys]
        total = sum(raw) or 1.0
        matrix.append([v / total for v in raw])  # normalise to %

    matrix = np.array(matrix)
    fig, ax = plt.subplots(figsize=(12, max(3, len(symbol_list) * 0.7)))
    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0)
    ax.set_xticks(range(len(labels)));   ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(symbol_list))); ax.set_yticklabels(symbol_list, fontsize=9)
    for i in range(len(symbol_list)):
        for j in range(len(labels)):
            ax.text(j, i, f'{matrix[i,j]:.1%}', ha='center', va='center', fontsize=7,
                    color='black' if matrix[i,j] < 0.25 else 'white')
    plt.colorbar(im, ax=ax, label='Normalised weight')
    ax.set_title('CCCi Indicator Weights per Symbol (Evolved)', fontsize=11, pad=12)
    plt.tight_layout()
    plt.savefig('/content/outputs/caribbean_capital/indicator_weights.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n📌 Key insight: Heavier weights = that indicator matters more for this market')

In [ ]:
# ─── 💰 $250 Account Growth Projection ───────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

STARTING_BALANCE = 250.0
PROP_TARGET_PCT  = 0.08   # 8% profit target for prop firm challenge

print(f"{'Symbol':<10} {'Net Return':>12} {'Final Balance':>14} {'Prop Target':>12} {'Status'}")
print('─' * 60)
for symbol, p in profiles.items():
    if symbol not in all_curves or not all_curves[symbol]:
        continue
    final_eq = all_curves[symbol][-1]
    net_ret  = final_eq - 1.0
    balance  = STARTING_BALANCE * final_eq
    prop_ok  = '✅ PASS' if net_ret >= PROP_TARGET_PCT and p.get('status') == 'PASS' else ('⚠️  Below' if p.get('status') != 'PASS' else '📈 Needs time')
    print(f"{symbol:<10} {net_ret:>+12.2%} ${balance:>12.2f} {PROP_TARGET_PCT:>11.0%}     {prop_ok}")

print()
print(f'Starting balance: ${STARTING_BALANCE}')
print(f'Prop firm challenge: +{PROP_TARGET_PCT:.0%} target, ≤5% daily DD, ≤10% total DD')

# Plot projected equity growth in $ terms
fig, axes = plt.subplots(1, min(4, len(all_curves)), figsize=(5 * min(4, len(all_curves)), 4))
if len(all_curves) == 1:
    axes = [axes]

for i, (symbol, curve) in enumerate(list(all_curves.items())[:4]):
    ax = axes[i] if len(all_curves) > 1 else axes[0]
    balance_curve = [STARTING_BALANCE * e for e in curve]
    color = '#00c853' if profiles.get(symbol, {}).get('status') == 'PASS' else '#ff6d00'
    ax.plot(balance_curve, color=color, linewidth=1.5)
    ax.axhline(y=STARTING_BALANCE * (1 + PROP_TARGET_PCT), color='blue', linestyle='--', alpha=0.7,
               label=f'Prop target ${STARTING_BALANCE*(1+PROP_TARGET_PCT):.0f}')
    ax.axhline(y=STARTING_BALANCE, color='gray', linestyle='--', alpha=0.5, label='Start')
    ax.fill_between(range(len(balance_curve)), STARTING_BALANCE, balance_curve, alpha=0.15, color=color)
    ax.set_title(f'{symbol}', fontsize=10)
    ax.set_ylabel('Balance ($)')
    ax.set_xlabel('Bars')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle(f'$250 Account Projection — Caribbean Capital EA', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/outputs/caribbean_capital/account_projection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 📋 Final Summary Table ──────────────────────────────────────────────────
import pandas as pd

rows = []
for symbol, p in profiles.items():
    rows.append({
        'Symbol': symbol,
        'Status': p['status'],
        'Bull Level': p['ccci_bull_level'],
        'Bear Level': p['ccci_bear_level'],
        'Stop ATR': p['stop_atr'],
        'Trail ATR': p['trail_atr'],
        'TP R:R': p['tp1_rr'],
        'Max Bars': p['max_holding_bars'],
        'Crossover': p['require_crossover'],
    })

summary_df = pd.DataFrame(rows).set_index('Symbol')
pd.set_option('display.max_columns', 20)
print("\n🏝️  Caribbean Capital — Evolved Parameters")
print(summary_df.to_string())
print(f"\nSave these parameters into your MT5 EA or evolved_profiles.json")